# Skill Graph Construction

This notebook constructs a skill graph from a dataset of skills with their knowledge points. The process includes:

1. Loading and embedding skill data
2. Deduplicating knowledge points
3. Constructing a graph based on co-occurrence
4. Processing the graph for further use

In [ ]:
from datasets import load_dataset
import numpy as np
from collections import defaultdict, Counter
import itertools
import networkx as nx
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the sentence transformer model
model = SentenceTransformer('./bge-large-en-v1.5/')

## Data Loading and Embedding

Load the skill dataset and create embeddings for the content.

In [ ]:
# Load the labeled skills file
file_path = './numina_100k_skills.jsonl'
dataset = load_dataset('json', data_files={'train': file_path})

# Embed the node content
content_ls = []
for data in dataset['train']:
    content_ls.append(data['content'])

content_embeds = model.encode(content_ls)
np.save('./numina_100k_content_embed.npy', content_embeds)

## Knowledge Point Extraction

Extract knowledge points from the dataset.

In [ ]:
# Get the knowledge points
kp_list = []
for data in dataset['train']:
    kp_list.append(data['knowledge points'])

## Knowledge Point Deduplication

Deduplicate similar knowledge points using cosine similarity.

In [ ]:
def dedup_list(lists_of_strings, threshold=0.8):
    all_strings = list(set(s for lst in lists_of_strings for s in lst))
    embeddings = model.encode(all_strings)
    similarity_matrix = cosine_similarity(embeddings)
    
    replace_map = {}
    for i, s1 in enumerate(all_strings):
        for j in range(i + 1, len(all_strings)):
            s2 = all_strings[j]
            if similarity_matrix[i][j] > threshold:
                if s2 not in replace_map:
                    replace_map[s2] = s1
    
    modified_lists = []
    for lst in lists_of_strings:
        modified_list = [replace_map.get(s, s) for s in lst]
        modified_lists.append(list(set(modified_list)))
    return modified_lists

# This step is time-consuming
dedup_kp_list = dedup_list(kp_list, threshold=0.9)

## Graph Construction

Construct the skill graph based on co-occurrence of knowledge points.

In [ ]:
string_count = Counter()
list_indices = defaultdict(list)

for index, lst in enumerate(dedup_kp_list):
    string_count.update(lst)
    for string in lst:
        list_indices[string].append(index)

edge_count = defaultdict(int)

for lst in dedup_kp_list:
    for str1, str2 in itertools.combinations(lst, 2):
        edge_count[frozenset([str1, str2])] += 1

total_lists = len(dedup_kp_list)

G = nx.Graph()

for string, count in string_count.items():
    G.add_node(string, count=count, list_indices=list_indices[string])

for edge, count in edge_count.items():
    str1, str2 = tuple(edge)  
    G.add_edge(str1, str2, probability=count / total_lists)

## Graph Saving and Loading

Save the constructed graph and demonstrate loading it back.

In [ ]:
# Save the graph
import pickle
with open("./graph_numina_100k.pkl", "wb") as f:
    pickle.dump(G, f)

# Load the graph
with open("./graph_numina_100k.pkl", "rb") as f:
    G = pickle.load(f)
    
num_nodes = G.number_of_nodes()
print("Number of nodes:", num_nodes)

num_edges = G.number_of_edges()
print("Number of edges:", num_edges)

## Node Attribute Processing

Process node attributes and create embeddings for knowledge points.

In [ ]:
node_kp_ls = []
count_ls = []
content_ls_ls = []
difficulty_ls = []

for node, attrs in G.nodes(data=True):
    content_ls_ls.append(attrs['list_indices'])
    node_kp_ls.append(node)
    count_ls.append(attrs['count'])

content_embeds = np.load('./numina_100k_content_embed_qwen25.npy')
mean_kp_embeds = []

for ls in content_ls_ls:
    # print(len(ls))
    # print(np.mean(content_embeds[ls],axis=0).shape)
    mean_kp_embeds.append(np.mean(content_embeds[ls],axis=0))

mean_kp_embeds_stacked = np.vstack(mean_kp_embeds)
np.save('./mean_kp_embeds_numina100k.npy', mean_kp_embeds_stacked)

## Adjacency Matrix Processing

Process the adjacency matrix with softmax transformations.

In [ ]:
adj_matrix_numpy = nx.to_numpy_array(G, weight='probability')

In [ ]:
import torch

adj_matrix = torch.tensor(adj_matrix_numpy, dtype=torch.float32)

def softmax_with_temperature(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than 0.")
    logits = logits / temperature
    max_logits = torch.max(logits, dim=-1, keepdim=True).values
    exp_logits = torch.exp(logits - max_logits)
    sum_exp_logits = torch.sum(exp_logits, dim=-1, keepdim=True)
    probabilities = exp_logits / sum_exp_logits
    return probabilities

cnt_weighted = softmax_with_temperature(torch.tensor(count_ls), temperature=10000)

In [ ]:
def softmax_with_nonzero_elements(matrix):
    flattened = matrix.flatten()
    non_zero_indices = flattened.nonzero(as_tuple=True)[0]
    non_zero_elements = flattened[non_zero_indices]
    
    print(non_zero_elements.max())
    print(non_zero_elements.min())
    print(non_zero_elements.max()/non_zero_elements.min())
    
    softmaxed_elements = softmax_with_temperature(non_zero_elements, temperature=0.05)
    
    print(softmaxed_elements.max())
    print(softmaxed_elements.min())
    print(softmaxed_elements.max()/softmaxed_elements.min())
    
    processed_flattened = torch.zeros_like(flattened)
    processed_flattened[non_zero_indices] = softmaxed_elements
    
    processed_adj_matrix = processed_flattened.reshape(matrix.shape)
    
    sums = torch.sum(processed_adj_matrix, dim=1)
    print(sums.max())
    print(sums.min())
    
    return processed_adj_matrix

processed_adj_matrix = softmax_with_nonzero_elements(adj_matrix)

## Final Graph Modification

Apply final modifications to the adjacency matrix and save the updated graph.

In [ ]:
for i in range(processed_adj_matrix.shape[0]):
    processed_adj_matrix[i][i] = cnt_weighted[i]

In [ ]:
zero_matrix = torch.zeros_like(processed_adj_matrix)
diagonal_matrix = torch.diag(torch.diag(processed_adj_matrix))

In [ ]:
G_modified = nx.from_numpy_array(processed_adj_matrix.numpy())

with open("./graph_numina_100k_new_adj_1.pkl", "wb") as f:
    pickle.dump(G_modified, f)

In [ ]:
# Demonstrate loading the modified graph
with open("./graph_numina_100k_qwen25_new_adj_1.pkl", "rb") as f:
    G_adj = pickle.load(f)

adj_matrix_numpy = nx.to_numpy_array(G_adj)